# Simple Average Ensemble: LeViT-128 + DenseNet121

Loads both trained models from Google Drive, generates predictions on the shared test set, and combines them with **simple averaging (soft voting)**:

`ensemble_probs = (preds_A + preds_B) / 2`

Results (classification report + confusion matrix) are saved to Drive under `thesis_ensemble_outputs/LeViT_DenseNet121_simple_avg/`.

In [1]:
# ============================================================
# STEP 1: Mount Drive and load the test set CSV
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, zipfile
import numpy as np
import pandas as pd
import tensorflow as tf

# --- Extract the image dataset zip (skip if already extracted this session) ---
split_zip_path = '/content/drive/MyDrive/split_dataset.zip'
if not os.path.exists('/content/content/split_dataset') and not os.path.exists('/content/split_dataset/test'):
    shutil.copy(split_zip_path, '/content/split_dataset.zip')
    with zipfile.ZipFile('/content/split_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print('Image dataset extracted!')

# --- Locate the test CSV (two possible source locations were used across notebooks) ---
csv_candidates = ['/content/drive/MyDrive/thesis_dataset_csv/test_data.csv']
csv_zip_path = '/content/drive/MyDrive/thesis_dataset_csv-20260716T043638Z-1-001.zip'
if not os.path.exists(csv_candidates[0]):
    if not os.path.exists('/content/csv_data'):
        with zipfile.ZipFile(csv_zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/csv_data')
    csv_candidates.append('/content/csv_data/thesis_dataset_csv/test_data.csv')

test_csv_path = next(p for p in csv_candidates if os.path.exists(p))
test_df = pd.read_csv(test_csv_path)
print('Using test CSV:', test_csv_path)

# --- Fix filepaths: try known extraction roots and keep the one that resolves ---
def fix_paths(df):
    original = df['filepath'].copy()
    candidates = [
        original,
        original.str.replace('/content/split_dataset', '/content/content/split_dataset', regex=False),
        original.str.replace('/content/split_dataset', '/content', regex=False),
    ]
    for cand in candidates:
        if os.path.exists(cand.iloc[0]):
            df = df.copy()
            df['filepath'] = cand
            return df
    raise FileNotFoundError('Could not resolve test image paths - check dataset extraction.')

test_df = fix_paths(test_df)
print('Sample path exists:', os.path.exists(test_df['filepath'].iloc[0]))
print('Test samples:', len(test_df))

class_names = sorted(test_df['label'].unique())
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}
y_true = test_df['label'].map(label_to_index).values

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

Mounted at /content/drive
Image dataset extracted!
Using test CSV: /content/csv_data/thesis_dataset_csv/test_data.csv
Sample path exists: True
Test samples: 2538


In [2]:
# ============================================================
# STEP 2: Load raw test images once (uint8, resized) - shared by both models
# ============================================================
def load_raw_images(filepaths, img_size=IMG_SIZE):
    images = []
    for fp in filepaths:
        img = tf.io.read_file(fp)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, img_size)
        img = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)
        images.append(img.numpy())
    return np.stack(images, axis=0)

X_raw = load_raw_images(test_df['filepath'].values)
print('Loaded test images:', X_raw.shape)

Loaded test images: (2538, 224, 224, 3)


In [3]:
# ============================================================
# Load Model A: LeViT-128
# ============================================================
!pip install -q keras-cv-attention-models
from keras_cv_attention_models import levit  # needed to deserialize the custom LeViT layer

model_A_path = '/content/drive/MyDrive/thesis_levit_outputs/best_levit_model.keras'
model_A = tf.keras.models.load_model(model_A_path, safe_mode=False)
print('LeViT-128 loaded')

# torch-style normalization is baked into the model graph via a Lambda layer
X_A = X_raw.astype('float32')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.1/191.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
[WARNING] Setting TF_USE_LEGACY_KERAS=1. Make sure this is ahead of importing tensorflow or keras.


TypeError: Could not locate class 'Functional'. Make sure custom classes and functions are decorated with `@keras.saving.register_keras_serializable()`. If they are already decorated, make sure they are all imported so that the decorator is run before trying to load them. Full object config: {'module': 'tf_keras.src.engine.functional', 'class_name': 'Functional', 'config': {'name': 'model', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_4'}, 'registered_name': None, 'name': 'input_4', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'Lambda', 'config': {'name': 'lambda', 'trainable': True, 'dtype': 'float32', 'function': ['4wEAAAAAAAAAAAAAAAIAAAADAAAA8yIAAACXAHwAdAAAAAAAAAAAAHoKAAB0AgAAAAAAAAAAegsA\nAFMAKQFOKQLaBG1lYW7aA3N0ZCkB2gF0cwEAAAAg+iAvdG1wL2lweWtlcm5lbF82MzgvNDA0OTUw\nMDAzNi5wedoIPGxhbWJkYT5yBgAAAAsAAABzDQA

In [ ]:
# ============================================================
# Load Model B: DenseNet121
# ============================================================

model_B_path = '/content/drive/MyDrive/thesis_densenet121_outputs/best_densenet121_model.keras'
model_B = tf.keras.models.load_model(model_B_path)
print('DenseNet121 loaded')

# preprocessing (densenet.preprocess_input) is baked into the model graph
X_B = X_raw.astype('float32')

In [ ]:
# ============================================================
# STEP 5: Get predictions from both models
# ============================================================
MODEL_A_NAME = 'LeViT-128'
MODEL_B_NAME = 'DenseNet121'

preds_A = model_A.predict(X_A, batch_size=BATCH_SIZE, verbose=1)
preds_B = model_B.predict(X_B, batch_size=BATCH_SIZE, verbose=1)

print('Model A predictions shape:', preds_A.shape)
print('Model B predictions shape:', preds_B.shape)

# Save raw prediction arrays too, so weighted/triple ensembles later don't
# need to rerun inference from scratch
pred_backup_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/LeViT_DenseNet121_simple_avg'
os.makedirs(pred_backup_folder, exist_ok=True)
np.save(f'{pred_backup_folder}/preds_A.npy', preds_A)
np.save(f'{pred_backup_folder}/preds_B.npy', preds_B)
np.save(f'{pred_backup_folder}/y_true.npy', y_true)

In [ ]:
# ============================================================
# STEP 6: Simple average ensemble (soft voting) + evaluation
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

ensemble_probs = (preds_A + preds_B) / 2.0
y_pred_ensemble = np.argmax(ensemble_probs, axis=1)

acc_A = accuracy_score(y_true, np.argmax(preds_A, axis=1))
acc_B = accuracy_score(y_true, np.argmax(preds_B, axis=1))
ensemble_acc = accuracy_score(y_true, y_pred_ensemble)

print(f'{MODEL_A_NAME} Test Accuracy:              {acc_A:.4f}')
print(f'{MODEL_B_NAME} Test Accuracy:              {acc_B:.4f}')
print(f'Ensemble (simple average) Test Accuracy: {ensemble_acc:.4f}')

report = classification_report(y_true, y_pred_ensemble, target_names=class_names)
print(report)

output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/LeViT_DenseNet121_simple_avg'
os.makedirs(output_folder, exist_ok=True)

with open(f'{output_folder}/classification_report.txt', 'w') as f:
    f.write(f'{MODEL_A_NAME} Accuracy: {acc_A:.4f}\n')
    f.write(f'{MODEL_B_NAME} Accuracy: {acc_B:.4f}\n')
    f.write(f'Ensemble Accuracy: {ensemble_acc:.4f}\n\n')
    f.write(report)

cm = confusion_matrix(y_true, y_pred_ensemble)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('LeViT-128 + DenseNet121 - Confusion Matrix (Simple Average Ensemble)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{output_folder}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nResults saved to: {output_folder}')